# Core NLP Principles Notebook

### Topics Covered:
1. Text Preprocessing
2. Tokenization
3. TF-IDF
4. Mini NLP Pipeline (Integration)
5. Embeddings
6. Named Entity Recognition (NER)


## Install Required Libraries

Before starting, we will is eth below Python libraries int his notebook:
- NLTK -> Classic NLP library which we will use it for tokenization, stopwords, stemming.
- spaCy -> Modern, fast NLP (NER, lemmatization, vectors)
- scikit-learn -> TF-IDF vectorizer
- sentence-transformers -> Modern embeddings
- TextBlob -> Simple sentiment analysis


In [13]:
# # Installing all required libraries

# !pip install nltk spacy scikit-learn sentence-transformers textblob
# !python -m spacy download en_core_web_sm                     # To download spaCy's English model
import sys
!{sys.executable} -m spacy download en_core_web_sm


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 4.5 MB/s eta 0:00:03
     ---- ----------------------------------- 1.3/12.8 MB 3.4 MB/s eta 0:00:04
     ------- -------------------------------- 2.4/12.8 MB 3.8 MB/s eta 0:00:03
     --------- ------------------------------ 3.1/12.8 MB 3.6 MB/s eta 0:00:03
     ------------ --------------------------- 3.9/12.8 MB 3.6 MB/s eta 0:00:03
     -------------- ------------------------- 4.7/12.8 MB 3.7 MB/s eta 0:00:03
     ---------------- ----------------------- 5.2/12.8 MB 3.6 MB/s eta 0:00:03
     ------------------ --------------------- 6.0/12.8 MB 3.5 MB/s eta 0:00:02
     -------------------- ------------------- 6.6/12.8 MB 3.5 MB/s eta 0:00:02
     ---------------------- ----------------- 7.1/12.8 MB 3.4 MB/s eta 0:00:02
     ------------------------ --------------- 7.9/12.8 MB 3.4 MB/s


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import nltk

# # Downloading NLTK data files needed for the NLP tasks thal we will perform below. (This will be commented because we only have to download it once.)
# nltk.download('punkt')           # For tokenization
# nltk.download('punkt_tab')       # Updated tokenizer data
# nltk.download('stopwords')       # Common words like 'the', 'is', 'in'
# nltk.download('wordnet')         # For lemmatization
# nltk.download('averaged_perceptron_tagger')  # POS tagging
# nltk.download('vader_lexicon')   # For sentiment analysis later

# print("NLTK data downloaded successfully!")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\siddhant.khobragade\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\siddhant.khobragade\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\siddhant.khobragade\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\siddhant.khobragade\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\siddhant.khobragade\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\siddhant.khobragade\AppData\

NLTK data downloaded successfully!


# SECTION 1: Text Preprocessing

## What is Text Preprocessing?

Imagine we have a sentence like:
    - "The quick brown fox!! Jumped over 3 lazy dogs... Running quickly!!"

Before any AI/ML model can understand this text, we need to clean it up. So this cleaning process is called Text Preprocessing.

Think of it like washing vegetables before cooking. Raw text has:
- Uppercase letters -> So this confuses models (`Dog` and `dog` are not same for a computer)
- Punctuation -> `!!`, `...` add no meaning
- Stopwords -> `the`, `is`, `a` are useless in most tasks
- Different word forms -> `running`, `ran`, `runs` all mean the same root action

### Steps in Text Preprocessing:
| Step | What it does | Example |
|------|-------------|--------|
| Lowercasing | Convert all text to lowercase | `Dog` -> `dog` |
| Punctuation Removal | Remove special characters | `hello!!` -> `hello` |
| Stopword Removal | Remove common useless words | `the cat` -> `cat` |
| Stemming | Cut word to its root form (rough) | `running` -> `run` |
| Lemmatization | Convert to proper dictionary root | `better` -> `good` |

In [3]:
# ---------------------------------------------------
# TOPIC: Lowercasing
# ---------------------------------------------------
# Why? Computers treat 'Apple' and 'apple' as different words.
# Lowercasing makes sure they're treated the same.

sample_text = "The QUICK Brown Fox Jumped Over The Lazy Dog!"
print("Original text:  ", sample_text)

# We use built-in .lower() to convert the text to lowercase.
lowercased_text = sample_text.lower()
print("After lowercasing:", lowercased_text)

Original text:   The QUICK Brown Fox Jumped Over The Lazy Dog!
After lowercasing: the quick brown fox jumped over the lazy dog!


In [6]:
# ---------------------------------------------------
# TOPIC: Punctuation Removal
# ---------------------------------------------------
# Why? Punctuation like !, ., ?? adds no meaning to most NLP tasks.
# So here We use Pythons regex to remove them.

import re  

text_with_punct = "Hello!! How are you?? I'm doing great... Let's go through NLP concepts!"
print("Before:", text_with_punct)

# [^a-zA-Z0-9\s] means: remove anything that is NOT a letter, number, or space
cleaned_text = re.sub(r'[^a-zA-Z0-9\s]', '', text_with_punct)
print("After punctuation removal:", cleaned_text)

Before: Hello!! How are you?? I'm doing great... Let's go through NLP concepts!
After punctuation removal: Hello How are you Im doing great Lets go through NLP concepts


In [8]:
# ---------------------------------------------------
# TOPIC: Stopword Removal
# ---------------------------------------------------
# What are stopwords?
# Words like 'the', 'is', 'in', 'a', 'an' appear in almost every sentence.
# They don't carry meaningful information for tasks like keyword extraction.
# We remove them to focus on the important words so that our inal model performs good.

from nltk.corpus import stopwords

# Load English stopwords
stop_words = set(stopwords.words('english'))

# Let's see what stopwords look like (first 20)
print("Sample stopwords:", list(stop_words)[:20])
print("Total stopwords in English:", len(stop_words))

Sample stopwords: ['who', "weren't", 'can', 'mustn', 'him', "mustn't", 'whom', "he'd", 'them', 'how', 'hasn', 'too', 'wouldn', "she'd", "we'd", 'y', 'off', "won't", 'between', 're']
Total stopwords in English: 198


In [ ]:
from nltk.tokenize import word_tokenize

# Now let's remove stopwords from a sentence
sentence = "Natural Language Processing is a very important field in artificial intelligence"

# Step 1: Tokenize (split into words)
words = word_tokenize(sentence.lower())  # Also lowercase here
print("All words:", words)

# Step 2: Remove stopwords likehere weare keeping only words that are not presnt in stop_words
filtered_words = [word for word in words if word not in stop_words]
print("After stopword removal:", filtered_words)

# (We can notice that : 'is', 'a', 'very', 'in' are removed. And only the important words are presnet now.)

All words: ['natural', 'language', 'processing', 'is', 'a', 'very', 'important', 'field', 'in', 'artificial', 'intelligence']
After stopword removal: ['natural', 'language', 'processing', 'important', 'field', 'artificial', 'intelligence']


In [11]:
# ---------------------------------------------------
# TOPIC: Stemming vs Lemmatization
# ---------------------------------------------------
# Both try to find the 'base form' of a word.
# The difference:
#   STEMMING  -> chops off word endings (fast but rough, may not give real words)
#   LEMMATIZATION -> uses a dictionary to find the actual root word (slower but accurate)

# Example:
#   'studies' -> Stemming = 'studi' whereas  Lemmatization = 'study'
#   'better' -> Stemming = 'better' whereas  Lemmatization = 'good'
#   'running' -> Stemming = 'run'  whereas Lemmatization = 'run'

from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

# Initialize
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Test words
test_words = ['running', 'studies', 'better', 'wolves', 'playing', 'happily', 'geese']

print(f"{'Word':<12} {'Stemmed':<15} {'Lemmatized':<15}")
print("-" * 42)
for word in test_words:
    stemmed = stemmer.stem(word)           # Stemming
    lemmatized = lemmatizer.lemmatize(word)  # Lemmatization
    print(f"{word:<12} {stemmed:<15} {lemmatized:<15}")

print("\n We can notice here : Studies word got converted correctly while using Lemmatization as 'study' where as in Stemming we got 'studi' whic is incorrect also 'geese' -> Lemmatized = 'goose' (correct!), Stemming can't do this.")

Word         Stemmed         Lemmatized     
------------------------------------------
running      run             running        
studies      studi           study          
better       better          better         
wolves       wolv            wolf           
playing      play            playing        
happily      happili         happily        
geese        gees            goose          

 We can notice here : Studies word got converted correctly while using Lemmatization as 'study' where as in Stemming we got 'studi' whic is incorrect also 'geese' -> Lemmatized = 'goose' (correct!), Stemming can't do this.


In [ ]:
# ---------------------------------------------------
# PUTTING IT ALL TOGETHER: Full Preprocessing Pipeline
# ---------------------------------------------------
# Now let's combine all steps into one reusable function

import re
import spacy
from nltk.corpus import stopwords

# Load spaCy's English model
nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Full preprocessing pipeline:
    1. Lowercase
    2. Remove punctuation
    3. Lemmatize using spaCy
    4. Remove stopwords
    """
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # Step 3: Use spaCy for lemmatization
    doc = nlp(text)  # spaCy processes the text
    
    # Step 4: Keep only meaningful, lemmatized tokens
    tokens = [
        token.lemma_          # Get the root/base form of the word
        for token in doc
        if token.text not in stop_words   # Remove stopwords
        and not token.is_punct            # Remove punctuation tokens
        and not token.is_space            # Remove whitespace tokens
        and len(token.text) > 2           # Remove very short words (like 'a', 'an')
    ]
    
    return tokens

# Now lets test it!
sample = "The scientists were studying the effects of machine learning on natural language processing tasks."
print("Original:", sample)
print("Processed:", preprocess_text(sample))

Original: The scientists were studying the effects of machine learning on natural language processing tasks.
Processed: ['scientist', 'study', 'effect', 'machine', 'learn', 'natural', 'language', 'processing', 'task']


# SECTION 2: Tokenization

## What is Tokenization?

**Tokenization** is the process of breaking text into smaller units called **tokens**.

Think of it like cutting a sentence into individual Lego pieces so a computer can analyze them one by one.

**Types of Tokenization:**
| Type | What it does | Example |
|------|-------------|--------|
| Word Tokenization | Split into individual words | `"I love NLP"` → `['I', 'love', 'NLP']` |
| Sentence Tokenization | Split into sentences | `"Hi. How are you?"` → `['Hi.', 'How are you?']` |
| Subword Tokenization | Split words into smaller pieces | `"unhappy"` → `['un', 'happy']` |

### Why does Tokenization matter?
Every AI/NLP model starts by tokenizing the input. Before any analysis happens, the text must be broken into tokens. It's the **first step** in every NLP pipeline.

### Subword Tokenization (Concept Only)
Modern models like **BERT** and **GPT** use subword tokenization. For example:
- `"unhappiness"` → `['un', '##happi', '##ness']` (BERT style)
- This helps handle unknown or rare words by breaking them into known pieces.
- You don't need to implement this yourself — transformer libraries do it automatically.

In [15]:
# ---------------------------------------------------
# TOPIC: Word Tokenization with NLTK
# ---------------------------------------------------

from nltk.tokenize import word_tokenize, sent_tokenize

text = "Natural Language Processing helps computers understand human language. It is used in chatbots, search engines, and translation."

# Word Tokenization — splits by words
word_tokens = word_tokenize(text)
print("Word Tokens (NLTK):")
print(word_tokens)
print(f"Total tokens: {len(word_tokens)}")

Word Tokens (NLTK):
['Natural', 'Language', 'Processing', 'helps', 'computers', 'understand', 'human', 'language', '.', 'It', 'is', 'used', 'in', 'chatbots', ',', 'search', 'engines', ',', 'and', 'translation', '.']
Total tokens: 21


In [16]:
# ---------------------------------------------------
# TOPIC: Sentence Tokenization with NLTK
# ---------------------------------------------------
# Sometimes we want to split by sentences, not words.
# This is useful for tasks like summarization.

sentence_tokens = sent_tokenize(text)
print("Sentence Tokens (NLTK):")
for i, sentence in enumerate(sentence_tokens, 1):
    print(f"  Sentence {i}: {sentence}")

Sentence Tokens (NLTK):
  Sentence 1: Natural Language Processing helps computers understand human language.
  Sentence 2: It is used in chatbots, search engines, and translation.


In [17]:
# ---------------------------------------------------
# TOPIC: Tokenization with spaCy
# ---------------------------------------------------
# spaCy is smarter than NLTK — it handles edge cases better.
# It also gives extra info per token (like POS tags, entity types, etc.)

import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp(text)  # spaCy processes the entire text

print("Tokens from spaCy:")
print([token.text for token in doc])  # Get text of each token

print("\n--- Bonus: Extra info spaCy gives us per token ---")
print(f"{'Token':<15} {'Lemma':<15} {'POS':<10} {'Is Stopword':<12}")
print("-" * 52)
for token in list(doc)[:12]:  # Show first 12 tokens
    print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<10} {str(token.is_stop):<12}")

# POS = Part of Speech (NOUN, VERB, ADJ, etc.)
# is_stop = Is this a stopword? True/False

Tokens from spaCy:
['Natural', 'Language', 'Processing', 'helps', 'computers', 'understand', 'human', 'language', '.', 'It', 'is', 'used', 'in', 'chatbots', ',', 'search', 'engines', ',', 'and', 'translation', '.']

--- Bonus: Extra info spaCy gives us per token ---
Token           Lemma           POS        Is Stopword 
----------------------------------------------------
Natural         Natural         PROPN      False       
Language        Language        PROPN      False       
Processing      processing      NOUN       False       
helps           help            VERB       False       
computers       computer        NOUN       False       
understand      understand      VERB       False       
human           human           ADJ        False       
language        language        NOUN       False       
.               .               PUNCT      False       
It              it              PRON       True        
is              be              AUX        True        
used    

In [ ]:
# ---------------------------------------------------
# COMPARISON: NLTK vs spaCy tokenization on tricky cases
# ---------------------------------------------------
# Let's see how they handle contractions and special cases

tricky_text = "Visit https://www.example.com/path?q=1&lang=en or email me@company.org for details!!!"

# NLTK
nltk_tokens = word_tokenize(tricky_text)
print("NLTK:", nltk_tokens)

# spaCy
spacy_doc = nlp(tricky_text)
spacy_tokens = [token.text for token in spacy_doc]
print("spaCy:", spacy_tokens)


# (Notice how it handles 'https://www.example.com/path?q=1&lang=en' and email 'me@company.org' differently)
# (SpaCy treats it as one token, while NLTK splits it into multiple parts)
# To summarixe the comparison:
    # For raw tokenization of clean text → nearly identical
    # For URLs, emails, special strings → spaCy wins
    # For linguistic metadata in one pass → spaCy wins by a large margin
    # For speed on simple tasks with no model loading → NLTK wins

NLTK: ['Visit', 'https', ':', '//www.example.com/path', '?', 'q=1', '&', 'lang=en', 'or', 'email', 'me', '@', 'company.org', 'for', 'details', '!', '!', '!']
spaCy: ['Visit', 'https://www.example.com/path?q=1&lang=en', 'or', 'email', 'me@company.org', 'for', 'details', '!', '!', '!']


# SECTION 3: TF-IDF (Term Frequency - Inverse Document Frequency)

## What is TF-IDF?

**TF-IDF** is a way to figure out **how important a word is** in a document - compared to all other documents.

### Intuition with an Example:
Imagine you have 3 news articles:
- Article 1: About *cricket*
- Article 2: About *football*
- Article 3: About *cricket*

+ The word **"the"** appears in ALL 3 articles -> Not important (low TF-IDF)
+ The word **"wicket"** appears only in cricket articles -> Very important for those articles (high TF-IDF)

### The Formula (Don't worry about math - just the idea):
```
TF-IDF = TF × IDF

TF  (Term Frequency)         = How often a word appears in THIS document
IDF (Inverse Document Freq)  = How rare the word is across ALL documents
                               (Rare = higher score)
```

**High TF-IDF** -> Word is frequent in THIS document but rare overall -> **Important keyword!**

### Why TF-IDF > Simple Word Counts?
Simple counting gives high score to `the`, `is`, `a` (useless words).  
TF-IDF penalizes common words and rewards unique, relevant words.

In [23]:
# ---------------------------------------------------
# TOPIC: TF-IDF using scikit-learn
# ---------------------------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Let's create a small collection of documents (called a 'corpus')
# Each string is one document
documents = [
    "Machine learning is a type of artificial intelligence",
    "Deep learning is a subset of machine learning",
    "Natural language processing uses machine learning",
    "Artificial intelligence includes machine learning and deep learning"
]

# Create the TF-IDF vectorizer
# stop_words='english' → automatically remove common English words
vectorizer = TfidfVectorizer(stop_words='english')

# Fit and transform: Learn from the corpus + convert to numbers
tfidf_matrix = vectorizer.fit_transform(documents)

# Get feature names (the vocabulary — all unique words)
feature_names = vectorizer.get_feature_names_out()

# Convert to a readable DataFrame
df = pd.DataFrame(
    tfidf_matrix.toarray(),   # Convert sparse matrix to regular array
    columns=feature_names,
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("TF-IDF Matrix (each row = a document, each column = a word):")
print(df.round(3))  # Round to 3 decimal places for readability

TF-IDF Matrix (each row = a document, each column = a word):
       artificial   deep  includes  intelligence  language  learning  machine  \
Doc 1       0.472  0.000     0.000         0.472     0.000     0.313    0.313   
Doc 2       0.000  0.456     0.000         0.000     0.000     0.604    0.302   
Doc 3       0.000  0.000     0.000         0.000     0.469     0.245    0.245   
Doc 4       0.384  0.384     0.486         0.384     0.000     0.508    0.254   

       natural  processing  subset   type   uses  
Doc 1    0.000       0.000   0.000  0.599  0.000  
Doc 2    0.000       0.000   0.579  0.000  0.000  
Doc 3    0.469       0.469   0.000  0.000  0.469  
Doc 4    0.000       0.000   0.000  0.000  0.000  


In [28]:
# ---------------------------------------------------
# Practical Use: Finding Top Keywords using TF-IDF
# ---------------------------------------------------
# This is exactly what a 'keyword extractor' does!

def get_top_keywords(text, top_n=5):
    """
    Given a piece of text, return the top N keywords using TF-IDF.
    Works best when you have multiple documents, but we simulate it here.
    """
    # We need at least one document; we'll use the text itself
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform([text])
    
    # Get the TF-IDF scores for all words
    feature_names = vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]  # Get scores for the first (only) document
    
    # Sort words by their TF-IDF score (highest first)
    word_scores = sorted(zip(feature_names, scores), key=lambda x: x[1], reverse=True)
    
    return word_scores[:top_n]

# Test with a paragraph
article = """
Artificial intelligence is transforming industries worldwide. Machine learning algorithms 
are being used in healthcare to detect diseases, in finance to detect fraud, and in 
transportation for autonomous vehicles. Deep learning neural networks have shown 
remarkable performance in image recognition and natural language processing tasks.
"""

print("Input text:")
print(article.strip())
print("\nTop 5 Keywords (by TF-IDF score):")
for word, score in get_top_keywords(article, top_n=5):
    print(f"  '{word}' → score: {score:.4f}")

Input text:
Artificial intelligence is transforming industries worldwide. Machine learning algorithms 
are being used in healthcare to detect diseases, in finance to detect fraud, and in 
transportation for autonomous vehicles. Deep learning neural networks have shown 
remarkable performance in image recognition and natural language processing tasks.

Top 5 Keywords (by TF-IDF score):
  'detect' → score: 0.3381
  'learning' → score: 0.3381
  'algorithms' → score: 0.1690
  'artificial' → score: 0.1690
  'autonomous' → score: 0.1690


# SECTION 4: Mini NLP Pipeline (Integration)

## What is an NLP Pipeline?

An **NLP Pipeline** is a sequence of steps that takes raw text as input and produces useful information as output.

Think of it like an assembly line in a factory:
```
Raw Text → Clean → Tokenize → TF-IDF → Top Keywords
```

This integration step is important because:
1. It shows you understand how the pieces fit together
2. It's a reusable, production-ready component
3. This is exactly how real NLP systems are built

In [29]:
# ---------------------------------------------------
# TOPIC: Full Mini NLP Pipeline
# Input Text -> Clean -> Tokenize -> TF-IDF -> Top Keywords
# ---------------------------------------------------

import re
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words('english'))

def nlp_pipeline(text, top_n=7):
    """
    Complete NLP pipeline:
    Step 1: Preprocess (lowercase, remove punctuation)
    Step 2: Tokenize using spaCy
    Step 3: Remove stopwords + lemmatize
    Step 4: TF-IDF to extract top keywords
    
    Returns: dict with each step's result
    """
    results = {}
    
    # ---- Step 1: Preprocess ----
    cleaned = text.lower()
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', cleaned)
    results['step1_cleaned'] = cleaned
    
    # ---- Step 2 & 3: Tokenize + Lemmatize + Remove stopwords ----
    doc = nlp(cleaned)
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop          # Not a stopword
        and not token.is_punct        # Not punctuation
        and len(token.text) > 2       # Not very short
        and token.is_alpha            # Only actual words (no numbers)
    ]
    results['step2_tokens'] = tokens
    
    # Join tokens back into a string for TF-IDF
    processed_text = ' '.join(tokens)
    
    # ---- Step 4: TF-IDF ----
    if len(tokens) > 0:
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform([processed_text])
        feature_names = vectorizer.get_feature_names_out()
        scores = tfidf_matrix.toarray()[0]
        
        # Sort by score and get top keywords
        word_scores = sorted(zip(feature_names, scores), key=lambda x: x[1], reverse=True)
        results['step3_keywords'] = word_scores[:top_n]
    
    return results


# --- TEST THE PIPELINE ---
input_text = """
The rise of generative AI and large language models like GPT has revolutionized 
how developers build applications. These models are trained on massive datasets 
and can generate human-like text, answer questions, and write code. Prompt engineering 
is the skill of crafting effective inputs to guide these AI models.
"""

print("=" * 60)
print("RUNNING NLP PIPELINE")
print("=" * 60)

output = nlp_pipeline(input_text)

print("\nINPUT TEXT:")
print(input_text.strip())

print("\nStep 1 — Cleaned Text:")
print(output['step1_cleaned'][:100], "...")

print("\nStep 2 — Tokens (lemmatized, stopwords removed):")
print(output['step2_tokens'])

print("\nStep 3 — Top Keywords (by TF-IDF):")
for word, score in output['step3_keywords']:
    print(f"  '{word}' → {score:.4f}")

RUNNING NLP PIPELINE

INPUT TEXT:
The rise of generative AI and large language models like GPT has revolutionized 
how developers build applications. These models are trained on massive datasets 
and can generate human-like text, answer questions, and write code. Prompt engineering 
is the skill of crafting effective inputs to guide these AI models.

Step 1 — Cleaned Text:

the rise of generative ai and large language models like gpt has revolutionized 
how developers bui ...

Step 2 — Tokens (lemmatized, stopwords removed):
['rise', 'generative', 'large', 'language', 'model', 'like', 'gpt', 'revolutionize', 'developer', 'build', 'application', 'model', 'train', 'massive', 'dataset', 'generate', 'humanlike', 'text', 'answer', 'question', 'write', 'code', 'prompt', 'engineering', 'skill', 'craft', 'effective', 'input', 'guide', 'model']

Step 3 — Top Keywords (by TF-IDF):
  'model' → 0.5000
  'answer' → 0.1667
  'application' → 0.1667
  'build' → 0.1667
  'code' → 0.1667
  'craft' → 0.1

# SECTION 5: Embeddings

## What are Embeddings?

**Embeddings** are a way to represent words (or sentences) as **numbers in a list** (called a vector), where **similar meanings result in similar numbers**.

### The Problem with TF-IDF:
TF-IDF treats each word as completely separate. It doesn't understand:
- `king` and `queen` are related
- `happy` and `joyful` are similar
- `car` and `automobile` mean the same thing

### How Embeddings Solve This:
Each word is converted to a vector (list of numbers) where:
- Words with similar meanings -> vectors that are **close to each other** in space
- Unrelated words -> vectors that are **far from each other**

```
TF-IDF (Sparse):    [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, ...]  (mostly zeros)
Embedding (Dense):  [0.23, -0.54, 0.88, 0.12, -0.33, ...]  (every value matters)
```

### Famous Example (Word2Vec):
```
vector('king') - vector('man') + vector('woman') ≈ vector('queen')
```
The model **learns relationships** between words from data!

### Why important ?
Embeddings are the foundation of:
- Semantic search
- Chatbots
- Document similarity

In [31]:
# ---------------------------------------------------
# TOPIC: Word Embeddings using spaCy
# ---------------------------------------------------
# spaCy's medium/large model has built-in word vectors.
# The 'sm' (small) model has limited vectors, but let's try.

import spacy
import numpy as np

nlp = spacy.load("en_core_web_sm")

# Get the vector for a single word
word = nlp("king")
print(f"Vector for 'king' (first 10 values): {word.vector[:10]}")
print(f"Vector shape (dimensions): {word.vector.shape}")
print("\nEach word is represented as a list of 96 numbers (in this model).")
print("   Bigger models use 300 or even 768/1536 dimensions.")

Vector for 'king' (first 10 values): [-1.0567218  -0.1372893  -0.08070132 -0.0436684  -0.12789571 -0.3033324
  1.8608588   0.7104616  -0.00786653  0.8059206 ]
Vector shape (dimensions): (96,)

Each word is represented as a list of 96 numbers (in this model).
   Bigger models use 300 or even 768/1536 dimensions.


In [32]:
# ---------------------------------------------------
# TOPIC: Word Similarity using Embeddings
# ---------------------------------------------------
# Two words are 'similar' if their vectors point in the same direction.
# We measure this with 'cosine similarity' (value between 0 and 1).
# 1.0 = identical, 0.0 = completely different

# Compare word similarity using spaCy
word_pairs = [
    ("king", "queen"),
    ("cat", "dog"),
    ("happy", "joyful"),
    ("car", "airplane"),
    ("king", "banana"),
]

print(f"{'Word 1':<12} {'Word 2':<12} {'Similarity Score':<20} {'Relationship'}")
print("-" * 65)

for w1, w2 in word_pairs:
    token1 = nlp(w1)
    token2 = nlp(w2)
    similarity = token1.similarity(token2)  # spaCy computes cosine similarity
    
    # Label the relationship
    if similarity > 0.7:
        relationship = "Very Similar"
    elif similarity > 0.4:
        relationship = "Somewhat Related"
    else:
        relationship = "Not Related"
    
    print(f"{w1:<12} {w2:<12} {similarity:.4f}{'':>12} {relationship}")

Word 1       Word 2       Similarity Score     Relationship
-----------------------------------------------------------------


C:\Users\siddhant.khobragade\AppData\Local\Temp\ipykernel_20884\2582908004.py:23: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  similarity = token1.similarity(token2)  # spaCy computes cosine similarity


king         queen        0.4220             Somewhat Related
cat          dog          0.7423             Very Similar
happy        joyful       0.6471             Somewhat Related
car          airplane     0.5085             Somewhat Related
king         banana       0.5395             Somewhat Related


In [33]:
# ---------------------------------------------------
# TOPIC: Sentence Embeddings using sentence-transformers
# ---------------------------------------------------
# sentence-transformers is MORE POWERFUL than spaCy for sentence-level similarity.
# This is what's used in real RAG systems and semantic search!

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load a pre-trained model (downloads ~90MB on first run)
# 'all-MiniLM-L6-v2' is small, fast, and good quality
print("Loading sentence transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

d:\Work\Learning\Topics_by_Abhay_Sir\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentence transformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3650.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


In [35]:
# Create sentence embeddings
sentences = [
    "I love programming in Python.",          # S1
    "Python is my favorite coding language.",  # S2 — very similar to S1
    "The weather is sunny today.",            # S3 — unrelated
    "Machine learning models need data.",     # S4
    "Deep learning requires large datasets.", # S5 — similar to S4
]

# Encode sentences to vectors
# Each sentence becomes a 384-dimensional vector
embeddings = model.encode(sentences)

print(f"Embedding shape per sentence: {embeddings[0].shape}")
print(f"Total embeddings: {embeddings.shape}")

# Now compute similarity between S1 and all others
query_sentence = sentences[0]
query_embedding = embeddings[0].reshape(1, -1)  # Reshape for sklearn

print(f"\nQuery: '{query_sentence}'")
print("\nSimilarity with all sentences:")
print("-" * 60)

for i, sentence in enumerate(sentences):
    # Compute cosine similarity between query and this sentence
    sim = cosine_similarity(query_embedding, embeddings[i].reshape(1, -1))[0][0]
    print(f"  [{sim:.4f}] {sentence}")

Embedding shape per sentence: (384,)
Total embeddings: (5, 384)

Query: 'I love programming in Python.'

Similarity with all sentences:
------------------------------------------------------------
  [1.0000] I love programming in Python.
  [0.8561] Python is my favorite coding language.
  [0.0175] The weather is sunny today.
  [0.1327] Machine learning models need data.
  [0.1634] Deep learning requires large datasets.


In [37]:
# ---------------------------------------------------
# TOPIC: Simple Semantic Search using Embeddings
# ---------------------------------------------------
# This is the CORE idea behind RAG!
# User asks a question -> Find the most similar document -> Return it as context

# Our 'knowledge base' — could be paragraphs from a PDF, website, etc.
knowledge_base = [
    "Python is a high-level programming language known for simplicity.",
    "Machine learning is a subset of artificial intelligence.",
    "The Eiffel Tower is located in Paris, France.",
    "Transformers are deep learning models used for NLP tasks.",
    "RAG stands for Retrieval Augmented Generation.",
    "India won the 2011 ICC Cricket World Cup.",
]

# Encode all documents in the knowledge base
kb_embeddings = model.encode(knowledge_base)

def semantic_search(query, top_k=2):
    """Find the most relevant documents for a given query."""
    # Encode the query
    query_embedding = model.encode([query])
    
    # Compute similarity with all documents
    similarities = cosine_similarity(query_embedding, kb_embeddings)[0]
    
    # Sort by similarity (highest first)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    print(f"\n Query: '{query}'")
    print("Most relevant documents:")
    for idx in top_indices:
        print(f"  [{similarities[idx]:.4f}] {knowledge_base[idx]}")

# Test semantic search
semantic_search("What is deep learning?")
semantic_search("Tell me about the Eiffel Tower")
semantic_search("How to code in Python?")


 Query: 'What is deep learning?'
Most relevant documents:
  [0.5403] Machine learning is a subset of artificial intelligence.
  [0.5118] Transformers are deep learning models used for NLP tasks.

 Query: 'Tell me about the Eiffel Tower'
Most relevant documents:
  [0.7960] The Eiffel Tower is located in Paris, France.
  [0.0873] Transformers are deep learning models used for NLP tasks.

 Query: 'How to code in Python?'
Most relevant documents:
  [0.6067] Python is a high-level programming language known for simplicity.
  [0.1420] Machine learning is a subset of artificial intelligence.


---
# SECTION 6: Named Entity Recognition (NER)


## What is NER?

**Named Entity Recognition (NER)** is the task of automatically identifying and classifying **named entities** in text.

A **named entity** is a real-world object with a name, like:
- A person → `Sundar Pichai`
- A company → `Google`
- A location → `New York`
- A date → `March 2024`
- A money amount → `$5 million`

### How does NER work?
A trained NER model reads text and labels each word with an entity type:

```
"Elon Musk founded Tesla in 2003 in California."
   PERSON          ORG    DATE     GPE
```

### Real-world uses:
- News analysis (who, what, where?)
- Resume parsing (extract name, company, skills)
- Medical records (extract patient names, diseases, medications)
- Financial analysis (extract company names, stock prices)

In [38]:
# ---------------------------------------------------
# TOPIC: Named Entity Recognition with spaCy
# ---------------------------------------------------

import spacy

nlp = spacy.load("en_core_web_sm")

# Sample text with multiple entity types
news_text = """
Apple Inc. CEO Tim Cook announced on Tuesday that the company will invest 
$10 billion in India over the next five years. The announcement was made 
in New Delhi during a meeting with Prime Minister Narendra Modi. 
Apple's stock rose 2.5% on the NASDAQ after the announcement.
"""

# Process with spaCy
doc = nlp(news_text)

print("Named Entities found:")
print("-" * 50)
print(f"{'Entity Text':<25} {'Label':<12} {'Meaning'}")
print("-" * 50)

# Entity label descriptions
label_meanings = {
    'PERSON': 'Person name',
    'ORG': 'Organization/Company',
    'GPE': 'Country/City/State',
    'DATE': 'Date or time',
    'MONEY': 'Money/Currency',
    'PERCENT': 'Percentage',
    'LOC': 'Location',
    'NORP': 'Nationality/Group',
    'PRODUCT': 'Product name'
}

for ent in doc.ents:  # .ents gives all recognized entities
    meaning = label_meanings.get(ent.label_, ent.label_)
    print(f"{ent.text:<25} {ent.label_:<12} {meaning}")

Named Entities found:
--------------------------------------------------
Entity Text               Label        Meaning
--------------------------------------------------
Apple Inc.                ORG          Organization/Company
Tim Cook                  PERSON       Person name
Tuesday                   DATE         Date or time
$10 billion               MONEY        Money/Currency
India                     GPE          Country/City/State
the next five years       DATE         Date or time
New Delhi                 GPE          Country/City/State
Narendra Modi             PERSON       Person name
Apple                     ORG          Organization/Company
2.5%                      PERCENT      Percentage
NASDAQ                    ORG          Organization/Company


In [39]:
# ---------------------------------------------------
# TOPIC: Grouping Entities by Type
# ---------------------------------------------------
# In real apps, you want to extract entities grouped by category

from collections import defaultdict

def extract_entities(text):
    """Extract and group all named entities from text."""
    doc = nlp(text)
    
    # Group entities by their label
    entity_groups = defaultdict(list)
    for ent in doc.ents:
        entity_groups[ent.label_].append(ent.text)
    
    return dict(entity_groups)

# Test with a longer text
article = """
Microsoft CEO Satya Nadella and Google's Sundar Pichai both attended the 
World Economic Forum in Davos, Switzerland in January 2024. Microsoft 
announced a $13 billion investment in OpenAI, while Google revealed plans 
to expand its AI research center in London, United Kingdom.
"""

entities = extract_entities(article)

print("Extracted Entities by Type:")
print("=" * 40)
for entity_type, entity_list in entities.items():
    # Remove duplicates while preserving order
    unique_entities = list(dict.fromkeys(entity_list))
    print(f"\n{entity_type}:")
    for entity in unique_entities:
        print(f"  -> {entity}")

Extracted Entities by Type:

ORG:
  -> Microsoft
  -> Google

PERSON:
  -> Satya Nadella
  -> Sundar Pichai

GPE:
  -> Davos
  -> Switzerland
  -> OpenAI
  -> AI
  -> London
  -> United Kingdom

DATE:
  -> January 2024

MONEY:
  -> $13 billion


In [40]:
# ---------------------------------------------------
# TOPIC: Visualizing NER (displacy)
# ---------------------------------------------------
# spaCy has a built-in visualizer called 'displacy'
# This renders highlighted entities directly in Jupyter!

from spacy import displacy

# Simple sentence for visualization
vis_text = "Elon Musk founded SpaceX in 2002 in Hawthorne, California with $100 million."

doc_vis = nlp(vis_text)

# Render in Jupyter Notebook
# This will show colored highlights for each entity type
displacy.render(doc_vis, style="ent", jupyter=True)

---
# FINAL SUMMARY: What You've Learned

| # | Topic | What it does | Tool Used |
|---|-------|-------------|----------|
| 1 | **Text Preprocessing** | Clean raw text (lowercase, remove noise, lemmatize) | NLTK, spaCy |
| 2 | **Tokenization** | Split text into words/sentences | NLTK, spaCy |
| 3 | **TF-IDF** | Find important keywords in documents | scikit-learn |
| 4 | **NLP Pipeline** | Combine all steps end-to-end | All above |
| 5 | **Embeddings** | Represent text as numbers (semantic meaning) | spaCy, sentence-transformers |
| 6 | **NER** | Extract people, places, orgs from text | spaCy |


In [ ]:
# ---------------------------------------------------
# We have used here all the bove topics that we have learned -
    # Use all 6 concepts in one Streamlit app: "Text Analyzer Tool"
    # - Input: Any text
    # - Output: Keywords (TF-IDF) + Entities (NER) + Similarity Search (Embeddings)
# ---------------------------------------------------

import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
from collections import defaultdict

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words('english'))
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def analyze_text(text):
    """
    Full text analysis combining:
    - TF-IDF keywords
    - Named entities
    - Semantic embedding (vector)
    """
    doc = nlp(text)
    
    # --- Keywords via TF-IDF ---
    cleaned = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    vectorizer = TfidfVectorizer(stop_words='english', max_features=10)
    tfidf = vectorizer.fit_transform([cleaned])
    keywords = vectorizer.get_feature_names_out().tolist()
    
    # --- Named Entities ---
    entities = defaultdict(list)
    for ent in doc.ents:
        entities[ent.label_].append(ent.text)
    
    # --- Sentence Embedding ---
    embedding = embed_model.encode([text])[0]
    
    return {
        "keywords": keywords,
        "entities": dict(entities),
        "embedding_dim": embedding.shape[0],
        "embedding_preview": embedding[:5].tolist()
    }


# --- TEST ---
test_input = """
Narendra Modi met with the CEO of Microsoft, Satya Nadella, in New Delhi last Monday.
They discussed expanding AI and cloud computing investments in India worth $3 billion.
The meeting was held at Hyderabad House and lasted approximately two hours.
"""

result = analyze_text(test_input)

print("=" * 55)
print(" TEXT ANALYSIS RESULTS")
print("=" * 55)
print(f"\n Top Keywords: {result['keywords']}")
print("\n  Named Entities:")
for etype, elist in result['entities'].items():
    print(f"  {etype}: {list(set(elist))}")
print(f"\n Embedding dimensions: {result['embedding_dim']}")
print(f"   First 5 values: {result['embedding_preview']}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4007.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 TEXT ANALYSIS RESULTS

 Top Keywords: ['ai', 'approximately', 'billion', 'ceo', 'cloud', 'computing', 'delhi', 'discussed', 'expanding', 'held']

  Named Entities:
  PERSON: ['Narendra Modi', 'Satya Nadella']
  ORG: ['Hyderabad House', 'Microsoft']
  GPE: ['New Delhi', 'AI', 'India']
  DATE: ['last Monday']
  MONEY: ['$3 billion']
  TIME: ['approximately two hours']

 Embedding dimensions: 384
   First 5 values: [-0.05533257871866226, -0.017104074358940125, -0.005010077729821205, 0.005821515340358019, -0.006267264951020479]
